# Fig. 2: Wigner functions

Exact steady state (QuantumToolbox.jl) of the cooling model after the adiabatic elimination of the excited levels, with the cooling atoms treated as a bosonic mode $s$ (Holstein–Primakoff) and truncated Fock spaces for $a$, $s$ and $b$. We compute the Wigner function of cavity mode $a$ at $N_c = 6162$ and $r = 1, 3, 6, 9$ dB (the markers in panel a).

**Output:** `../figures/panel_b_wigners.svg` (panel b).

In [ ]:
using QuantumToolbox
using CairoMakie
using WignerSymbols
using ProgressMeter
using LinearAlgebra

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)

    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Fock-space truncation of modes a, s and b
Ncav_a = 35
Ncav_s = 3
Ncav_b = 3

In [ ]:
num_r = 4
num_N = 1

# Rb87 level structure (angular frequencies, with the frequency in GHz)
gamma = 2*pi*5.75*1e-3
gamma_D2 = 2*pi*6.06*1e-3
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*6.834
w_s = w_t+2*pi*0.30544+2*pi*4.271677
w_r = w_t-2*pi*0.50906+2*pi*4.271677
w_u = w_t_D2-2*pi*4.271677

N_atom = [6162] # Number of cooling atoms N_c

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = [0.115,0.345,0.691,1.04] # 1, 3, 6 and 9 dB
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and the effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$. Arrays have shape `(num_r, num_N)`.

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav.+3*w_r.-w_s).-sqrt.(4*gr^2*N_atom'.+(w_cav.-w_r).^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom'.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom'*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)'./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

# Step 6: eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom'.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # Coefficient of the s's term

# Step 7: eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom').^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom')./((delta_lamb.^2).+(N_atom'.*dec/2).^2)

## Operators

In [ ]:
a = kron(destroy(Ncav_a),eye(Ncav_s),eye(Ncav_b))
s = kron(eye(Ncav_a),destroy(Ncav_s),eye(Ncav_b))
b = kron(eye(Ncav_a),eye(Ncav_s),destroy(Ncav_b))

## Steady state

In [ ]:
progress = Progress(num_r)

ss = Array{QuantumObject,1}(undef, num_r)

Threads.@threads for i in 1:num_r
    for j in 1:num_N
        H = w_c[j]*a'a + w_c_b[j]*b'b + w_0[i,j]*s's + chi[j]*a'a*s's + lambda_s[i,j]*sqrt(N_atom[j])*(a*s+a'*s') + lambda_r[i,j]*sqrt(N_atom[j])*(a'*s+a*s') + w_0_b[j]*s's + chi_b[j]*b'b*s's + g[j]*sqrt(N_atom[j])*(b*s'+b'*s)
        c_ops = [sqrt(kappa)*a, sqrt(kappa_b)*b]

        ss[i] = steadystate(H,c_ops)

        ProgressMeter.next!(progress)
    end
end

Sanity checks: truncation of the Fock spaces (`check_a`, `check_b`, `check_s` should stay close to 1), validity of the Holstein–Primakoff approximation (`check_HP` ≪ 1) and quadrature fluctuations `sq`.

In [ ]:
check_a = zeros(num_r)
check_b = zeros(num_r)
check_s = zeros(num_r)
check_NLC = zeros(num_r)
check_HP = zeros(num_r)
sq = zeros(num_r)

for i in 1:num_r
    X = a+a'
    check_a[i] = real(expect(a*a'-a'a,ss[i]))
    check_b[i] = real(expect(b*b'-b'b,ss[i]))
    check_s[i] = real(expect(s*s'-s's,ss[i]))
    check_NLC[i] = real(expect(s's,ss[i]))*chi_b[1]/kappa_b
    check_HP[i] = real(expect(s's,ss[i]))/N_atom[1]
    sq[i] = sqrt(real(variance(X,ss[i])))
end

(; check_a, check_b, check_s, check_NLC, check_HP, sq)

## Wigner functions and covariance ellipses

In [ ]:
function cov_mat(X, P, rho) # Covariance matrix of the quadratures X, P
    R = [X, P]
    V = zeros(2, 2)

    for i in 1:2
        for j in 1:2
            Rij = (R[i]*R[j] + R[j]*R[i]) / 2
            V[i, j] = real(expect(Rij, rho) - expect(R[i], rho) * expect(R[j], rho))
        end
    end

    return V
end

X = (a+a')/sqrt(2)
P = 1*im*(a-a')/sqrt(2)
theta = LinRange(0, 2*pi, 360)
circle = [cos.(theta)', sin.(theta)']

# Vacuum (shot noise) ellipse
vacuum = kron(fock(Ncav_a,0),fock(Ncav_s,0),fock(Ncav_b,0))
V = cov_mat(X,P,vacuum)
vals, vecs = eigen(V)
shot_noise = (vecs * Diagonal(sqrt.(vals)) * circle)

# Steady-state ellipses
ellipse = Array{Vector,2}(undef, num_r, 2)

for i in 1:num_r
    V = cov_mat(X,P,ss[i])
    vals, vecs = eigen(V)
    ellipse[i] = (vecs * Diagonal(sqrt.(vals)) * circle)
end

In [ ]:
xx = LinRange(-6,6,200)

wigs = [wigner(ptrace(ss[i],1),xx,xx) for i in 1:num_r]

## Panel (b)

In [ ]:
Fig = Figure(size=(800,400))
for i in 1:num_r
    ax = Axis(Fig[1, i];
        aspect = DataAspect(),
        limits = (-2,2,-5,5),xlabel=L"$x$",ylabel=L"$p$",
        xticklabelfont = "Cambria Math",
        yticklabelfont = "Cambria Math",
        xticksvisible = true,
        yticksvisible = true,
        xgridvisible = true,
        ygridvisible = true,
    )
    y = wigs[i]'
    heatmap!(ax,xx,xx,y;colormap=:bwr,colorrange=(-maximum(y),maximum(y)),interpolate=true, rasterize=true)
    lines!(ax,ellipse[i][1][:],ellipse[i][2][:],linestyle=:solid,color=:black)
    lines!(ax,shot_noise[1][:],shot_noise[2][:],linestyle=:dash,color=:black)
end
save(joinpath("..", "figures", "panel_b_wigners.svg"), Fig)
Fig